## ML Pipeline

**【Data Science Project】 Building a Machine Learning Pipeline for a Predictive Car Price Model with PySpark**

In [1]:
import os

# point java home to actual conda package reference
os.environ["JAVA_HOME"] = "/Users/andreasliistro/mambaforge/pkgs/openjdk-22.0.1-hbeb2e11_0/lib/jvm"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.sql.functions import isnan, when, count, col, lit
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder

# init spark session
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "4g").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/02 13:38:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/01/02 13:38:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59300)
Traceback (most recent call last):
  File "/Users/andreasliistro/mambaforge/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/andreasliistro/mambaforge/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Users/andreasliistro/mambaforge/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/andreasliistro/mambaforge/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/andreasliistro/mambaforge/lib/python3.10/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/Users/andreasliistro/mambaforge/lib/python3.10/site-packages/pyspark/accumulators.py", line 267, in poll
    if 

In [3]:
# read previous written csv data
data = spark.read.csv("ML-pipeLine-car-prediction/data/vehicles_cleaned.csv", header=True, inferSchema=True, multiLine=True)
# data = spark.read.csv("ML-pipeLine-car-prediction/data/vehicles.csv", header=True, inferSchema=True)

In [3]:
# show schema as reference
data.printSchema()

root
 |-- model: string (nullable = true)
 |-- id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- year: double (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- posting_date: string (nullable = true)



In [4]:
# show statisctics
data.describe().toPandas().transpose()

25/01/02 13:44:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,0,1,2,3,4
summary,count,mean,stddev,min,max
id,404770,7.311441030127712E9,4479145.211213261,7207408119,7317098055
price,404770,77149.20430614917,1.2504261601552494E7,0.0,3.736928711E9
year,403648,2011.5387614951642,8.892779822265293,1900.0,2022.0
manufacturer,391786,None,None,acura,volvo
model,399484,1769.0351324666776,1209.0341811193036,-,zl1 camaro
condition,238619,None,None,excellent,salvage
fuel,402435,None,None,diesel,other
odometer,400631,96590.75641176045,194646.41772511785,0.0,1.0E7
transmission,402295,None,None,automatic,other


### Feature Engineering

In [17]:
# replace null values for OneHotEncoder
data = data.withColumn("manufacturer", when(col("manufacturer").isNull(), lit("unknown")).otherwise(col("manufacturer")))
data = data.withColumn("condition", when(col("condition").isNull(), lit("unknown")).otherwise(col("condition")))
data = data.withColumn("fuel", when(col("fuel").isNull(), lit("unknown")).otherwise(col("fuel")))
data = data.withColumn("transmission", when(col("transmission").isNull(), lit("unknown")).otherwise(col("transmission")))

### Build the ML pipeline

Combining all numeric columns into feature vector. The column `price` is used as target column

In [18]:
manufacturer_indexer = StringIndexer(inputCol='manufacturer', outputCol='manufacturer_index')
manufacturer_encoder = OneHotEncoder(inputCol='manufacturer_index', outputCol='manufacturer_ohe')

condition_indexer = StringIndexer(inputCol='condition', outputCol='condition_index')
condition_encoder = OneHotEncoder(inputCol='condition_index', outputCol='condition_ohe')

fuel_indexer = StringIndexer(inputCol='fuel', outputCol='fuel_index')
fuel_encoder = OneHotEncoder(inputCol='fuel_index', outputCol='fuel_ohe')

transmission_indexer = StringIndexer(inputCol='transmission', outputCol='transmission_index')
transmission_encoder = OneHotEncoder(inputCol='transmission_index', outputCol='transmission_ohe')

# model_indexer = StringIndexer(inputCol='model', outputCol='model_index')
# model_encoder = OneHotEncoder(inputCol='model_index', outputCol='model_ohe')

#assemble all numeric columns into one vector of features
assembler = VectorAssembler(
    inputCols=[
    'manufacturer_ohe',
    'condition_ohe',
    'fuel_ohe',
    'transmission_ohe',
    # 'model_ohe',
    'year',
    'odometer',
  ],
  outputCol='Attributes',
  handleInvalid="skip"
)

#create a regressor to predict car price
regressor = RandomForestRegressor(featuresCol='Attributes', labelCol='price')

#create pipeline
pipeline = Pipeline(stages=[
  manufacturer_indexer,
  manufacturer_encoder,
  condition_indexer,
  condition_encoder,
  fuel_indexer,
  fuel_encoder,
  transmission_indexer,
  transmission_encoder,
  # model_indexer,
  # model_encoder,
  assembler,
  regressor
])

#save pipeline
pipeline.write().overwrite().save('pipeline')

### Perform cross-validation

In [23]:
#load pipeline
pipelineModel = Pipeline.load('pipeline')

#build paramgrid
paramGrid = ParamGridBuilder().addGrid(regressor.numTrees, [1, 500]).build()

#build crossvalidator
crossval = CrossValidator(estimator=pipelineModel,
                          estimatorParamMaps=paramGrid,
                          evaluator=RegressionEvaluator(labelCol='price'), #price is the column we want to predict
                          numFolds=10)

### Split data set into train / test

In [24]:
#train test split
train_data, test_data = data.randomSplit([0.8, 0.2], seed=123)

#fit
cvModel = crossval.fit(train_data)

#extract best model and view all the stages of the pipeline that our data went through
bestModel = cvModel.bestModel
for x in range(len(bestModel.stages)):
  print(bestModel.stages[x])

25/01/02 14:00:32 WARN DAGScheduler: Broadcasting large task binary with size 1621.4 KiB
25/01/02 14:00:39 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/01/02 14:01:02 WARN DAGScheduler: Broadcasting large task binary with size 1621.8 KiB
25/01/02 14:01:08 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/01/02 14:01:31 WARN DAGScheduler: Broadcasting large task binary with size 1623.1 KiB
25/01/02 14:01:37 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/01/02 14:02:00 WARN DAGScheduler: Broadcasting large task binary with size 1623.4 KiB
25/01/02 14:02:06 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/01/02 14:02:29 WARN DAGScheduler: Broadcasting large task binary with size 1621.6 KiB
25/01/02 14:02:36 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
25/01/02 14:02:57 WARN DAGScheduler: Broadcasting large task binary with size 1623.3 KiB
25/01/02 14:03:03 WARN DAGScheduler:

StringIndexerModel: uid=StringIndexer_78f5d3c4ee31, handleInvalid=error
OneHotEncoderModel: uid=OneHotEncoder_170e7c832822, dropLast=true, handleInvalid=error
StringIndexerModel: uid=StringIndexer_8dc8e2bf7a2b, handleInvalid=error
OneHotEncoderModel: uid=OneHotEncoder_bbdc63f5ee14, dropLast=true, handleInvalid=error
StringIndexerModel: uid=StringIndexer_eff0dd0325f7, handleInvalid=error
OneHotEncoderModel: uid=OneHotEncoder_6c9d854399b0, dropLast=true, handleInvalid=error
StringIndexerModel: uid=StringIndexer_ecc2725fafdd, handleInvalid=error
OneHotEncoderModel: uid=OneHotEncoder_6dc237838b49, dropLast=true, handleInvalid=error
VectorAssembler_06d20deb8576
RandomForestRegressionModel: uid=RandomForestRegressor_b8dc7867397a, numTrees=500, numFeatures=58


In [25]:
#transform the test set (use cvModel as it knows to pick the best model to use)
pred = cvModel.transform(test_data)
pred.select('price', 'prediction').show()

+-------+------------------+
|  price|        prediction|
+-------+------------------+
|18937.0|20160.867161157086|
|63990.0| 26425.09710051373|
|41990.0| 36022.16182430569|
|41990.0| 36022.16182430569|
|38990.0|  27412.4987736314|
|60990.0|25512.063146347064|
|38990.0|  27412.4987736314|
|35590.0| 27321.84437904128|
|64590.0|28003.447729268202|
| 3600.0|18799.330681295447|
|17500.0|15709.682895932061|
|59990.0|25512.063146347064|
|24990.0|18534.629319358042|
|33990.0|26613.488633058598|
|29990.0| 25205.32355070679|
|38990.0|  27412.4987736314|
|  750.0|26782.995850039813|
|38900.0|23561.750704082082|
|45590.0|46891.482883389595|
| 7499.0|16316.455612722706|
+-------+------------------+
only showing top 20 rows



In [26]:
#evaluate
eval = RegressionEvaluator(labelCol='price')

#get rmse
rmse = eval.evaluate(pred)

#get mse
mse = eval.evaluate(pred, {eval.metricName:'mse'})

#get mae
mae = eval.evaluate(pred, {eval.metricName:'mae'})

#get r2
r2 = eval.evaluate(pred, {eval.metricName:'r2'})

#print
print('RMSE: %3f' %rmse)
print('MSE: %3f' %mse)
print('MAE: %3f' %mae)
print('R2: %3f' %r2)

RMSE: 15248609.910887
MSE: 232520104214400.062500
MAE: 148059.855969
R2: -0.001256


25/01/02 14:31:10 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 982118 ms exceeds timeout 120000 ms
25/01/02 14:31:10 WARN SparkContext: Killing executors is not supported by current scheduler.
25/01/02 14:46:24 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$